In [1]:
# ============================================
# CELL 5: Enhanced ResumeMatchNet Model
# ============================================

class ResumeMatchNet:
    """Enhanced model with skill-based scoring"""
    
    def __init__(self):
        self.vectorizer = None
        self.skill_weight = 0.4  # Weight for skill matching
        self.text_weight = 0.6    # Weight for text similarity
    
    def preprocess_texts(self, resume_text, job_text):
        """Preprocess both texts"""
        return preprocess_document(resume_text), preprocess_document(job_text)
    
    def extract_features(self, resume_processed, job_processed):
        """Extract TF-IDF features"""
        # Combine for consistent vocabulary
        documents = [resume_processed, job_processed]
        
        # Initialize and fit vectorizer
        self.vectorizer = TfidfVectorizer(
            max_features=500,
            ngram_range=(1, 2),
            stop_words='english'
        )
        
        vectors = self.vectorizer.fit_transform(documents)
        return vectors[0], vectors[1]
    
    def compute_text_similarity(self, resume_vec, job_vec):
        """Compute cosine similarity between texts"""
        similarity = cosine_similarity(resume_vec, job_vec)
        return similarity[0][0]
    
    def compute_skill_score(self, resume_skills, job_skills):
        """Compute skill-based matching score"""
        if not job_skills:
            return 0.0
        
        matched = len(resume_skills.intersection(job_skills))
        total_job_skills = len(job_skills)
        
        # Score based on percentage of job skills matched
        return matched / total_job_skills if total_job_skills > 0 else 0
    
    def match(self, resume_text, job_text):
        """Complete matching pipeline"""
        # Extract skills
        resume_skills = extract_skills(resume_text)
        job_skills = extract_skills(job_text)
        
        # Preprocess texts
        resume_processed, job_processed = self.preprocess_texts(resume_text, job_text)
        
        # If preprocessing failed
        if not resume_processed or not job_processed:
            return {
                'match_score': 0,
                'skill_match_score': 0,
                'text_similarity': 0,
                'matched_skills': [],
                'missing_skills': list(job_skills),
                'resume_skills': list(resume_skills),
                'job_skills': list(job_skills)
            }
        
        # Extract features and compute similarity
        resume_vec, job_vec = self.extract_features(resume_processed, job_processed)
        text_similarity = self.compute_text_similarity(resume_vec, job_vec)
        
        # Compute skill score
        skill_score = self.compute_skill_score(resume_skills, job_skills)
        
        # Calculate final score (weighted combination)
        final_score = (self.text_weight * text_similarity + 
                      self.skill_weight * skill_score) * 100
        
        # Get skill match details
        matched_skills = resume_skills.intersection(job_skills)
        missing_skills = job_skills - resume_skills
        
        return {
            'match_score': round(final_score, 2),
            'skill_match_score': round(skill_score * 100, 2),
            'text_similarity': round(text_similarity * 100, 2),
            'matched_skills': list(matched_skills),
            'missing_skills': list(missing_skills),
            'resume_skills': list(resume_skills),
            'job_skills': list(job_skills),
            'skill_count': {
                'resume': len(resume_skills),
                'job': len(job_skills),
                'matched': len(matched_skills)
            }
        }

print("✅ ResumeMatchNet model ready!")

✅ ResumeMatchNet model ready!
